In [ ]:
import functools
import warnings

import botocore
import boto3
from iterpop import iterpop as ip
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd
import pandas as pd
from pandas.util import hash_pandas_object
from scipy import stats as scipy_stats
import seaborn as sns
from teeplot import teeplot as tp
from tqdm import tqdm

from dishpylib.pyhelpers import fit_control_t_distns

warnings.filterwarnings("ignore")


In [ ]:
from dishpylib.pyhelpers import print_runtime


In [ ]:
print_runtime()


In [ ]:
teeplot_subdir = "2026-02-04-cryptic-complexity"


In [ ]:
@functools.lru_cache
def get_control_t_distns( bucket, prefix, endeavor, stint ):

    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    control_competitions, = bucket_handle.objects.filter(
        Prefix=f'endeavor={endeavor}/{prefix}control-competitions/stage={2 + bool(prefix)}+what=collated/stint={stint}/',
    )

    control_df = pd.read_csv(
        f's3://{bucket}/{control_competitions.key}',
    )

    return fit_control_t_distns(control_df[
        control_df["Root ID"] == 1
    ].copy())


In [ ]:
def preprocess_competition_fitnesses(competitions_df, control_fits_df):
    print(len(competitions_df), "competitions to preprocess")
    print(len(control_fits_df), "control fits available")
    # preprocess data
    @functools.lru_cache
    def h0_fit(series):
        return ip.popsingleton(
            control_fits_df[control_fits_df["Series"] == series].to_dict(
                orient="records",
            )
        )

    competitions_df["p"] = competitions_df.apply(
        lambda row: scipy_stats.t.cdf(
            row["Fitness Differential"],
            h0_fit(row["genome series"])["Fit Degrees of Freedom"],
            loc=h0_fit(row["genome series"])["Fit Loc"],
            scale=h0_fit(row["genome series"])["Fit Scale"],
        ),
        axis=1,
    )
    competitions_df["Is Less Fit"] = competitions_df["p"] < 1.0 / 40
    competitions_df["Is More Fit"] = competitions_df["p"] > (1.0 -  1.0 / 40)
    competitions_df["Is Neutral"] = ~(
        competitions_df["Is Less Fit"] | competitions_df["Is More Fit"]
    )
    competitions_df["Relative Fitness"] = competitions_df.apply(
        lambda row: (
            "Significantly Advantageous"
            if row["Is More Fit"]
            else (
                "Significantly Deleterious" if row["Is Less Fit"] else "Neutral"
            )
        ),
        axis=1,
    )

    return competitions_df


# get data


In [ ]:
bucket = "prq49"
dfs = []
for prefix in [
    "cryptic-",
    # "",
]:
    if "step" in bucket:
        step = int(bucket.split("-step")[-1]) + 1
    else:
        step = 0
    if "restint" in bucket:
        kind = bucket.split("-")[3]
    else:
        kind = None
    s3_handle = boto3.resource(
        's3',
        region_name="us-east-2",
        config=botocore.config.Config(
            signature_version=botocore.UNSIGNED,
        ),
    )
    bucket_handle = s3_handle.Bucket(bucket)

    for stint in tqdm([10, 20, 30, 40, 50, 60, 70, 80, 90, 100]):
        try:
            series_profiles, = bucket_handle.objects.filter(
                Prefix=f'endeavor=16/{prefix}variant-competitions/stage=3+what=collated/stint={stint}/',
            )
            import warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
            control_fits_df = get_control_t_distns(bucket, prefix, 16, stint)
            df = pd.read_csv(
                f's3://{bucket}/{series_profiles.key}',
                compression='xz',
            )
            # df = df[df["Competition Series"] == 16005]
            # df = df.groupby([
            #     "genome variation"
            # ]).mean(numeric_only=True).reset_index()
            df["Stint"] = stint
            df["Series"] = df["Competition Series"]
            dfdigest = "{:x}".format( hash_pandas_object( df ).sum() )
            df = preprocess_competition_fitnesses(df, control_fits_df)
            assert "Series" in df.columns, df.columns

            df = df.copy()
            df["bucket"] = bucket
            df["variant"] = {"cryptic-": "skeleton", "": "wildtype"}[prefix]
            dfs.append(df)
        except Exception as e:
            print(e)
            print(f"Skipping {bucket=}, {stint=}, {prefix=}")


In [ ]:
df = pd.concat(dfs)


In [ ]:
pd.options.display.max_columns = None


In [ ]:
dfxx = df[
    df["Root ID"] == 1
].groupby(["Series", "Stint", "bucket", "variant"]).agg(
    {
        "Is More Fit": "sum",
        "Is Less Fit": "sum",
        "Is Neutral": "sum",
        "genome variation": "count",
    },
)
dfxx


In [ ]:
dfxx = dfxx.reset_index(drop=False)


In [ ]:
dfxx["epoch"] = np.minimum(dfxx["Stint"], 99) // 20


In [ ]:
dfxx["diff"] = dfxx["Is Less Fit"] - dfxx["Is More Fit"]


In [ ]:
sns.lineplot(
    data=dfxx,
    x="Stint",
    y="Is Less Fit",
    # hue="Series",
    estimator=np.mean,
)


In [ ]:
sns.lineplot(
    data=dfxx[
        dfxx["Is More Fit"] < 100
    ],
    x="Stint",
    y="Is Less Fit",
    # hue="Series",
    estimator=np.mean,
)


In [ ]:
sns.lineplot(
    data=dfxx,
    x="Stint",
    y="diff",
    # hue="Series",
    estimator=np.median,
)


In [ ]:
sns.lineplot(
    data=dfxx[dfxx["Series"] == 16005],
    x="Stint",
    y="Is Less Fit",
    # hue="Series",
    estimator=np.median,
)


In [ ]:
s3_handle = boto3.resource(
    "s3",
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket("prq49")

dfnone = pd.concat(
    [
        pd.read_csv(f"s3://prq49/{item.key}")
        for item in bucket_handle.objects.filter(
            Prefix=f"endeavor=16/external-competitions/stage=2+what=collated/",
        )
    ],
    ignore_index=True,
)
dfnone["kind"] = "none"
print(dfnone["Competition Stint"].unique())
print(dfnone["Competition Series"].unique())


In [ ]:
s3_handle = boto3.resource(
    "s3",
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket("prq49")

dfbio = pd.concat(
    [
        pd.read_csv(f"s3://prq49/{item.key}")
        for item in bucket_handle.objects.filter(
            Prefix=f"endeavor=16/external-competitions-focalbb-true/stage=2+what=collated/",
        )
    ],
    ignore_index=True,
)
dfbio["kind"] = "eco"
print(dfbio["Competition Stint"].unique())
print(dfbio["Competition Series"].unique())


In [ ]:
s3_handle = boto3.resource(
    "s3",
    region_name="us-east-2",
    config=botocore.config.Config(
        signature_version=botocore.UNSIGNED,
    ),
)
bucket_handle = s3_handle.Bucket("prq49")

dfabio = pd.concat(
    [
        pd.read_csv(f"s3://prq49/{item.key}")
        for item in bucket_handle.objects.filter(
            Prefix=f"endeavor=16/external-competitions-focalbb/stage=2+what=collated/",
        )
    ],
    ignore_index=True,
)
dfabio["kind"] = "self"
print(dfabio["Competition Stint"].unique())
print(dfabio["Competition Series"].unique())


In [ ]:
dfx = pd.concat([dfabio, dfbio, dfnone], ignore_index=True)
dfx["Fitness Differential Focal Sign"] = np.sign(
    dfx["Fitness Differential Focal"],
)
dfx["Series"] = dfx["Competition Series"]
dfx["Stint"] = dfx["Competition Stint"]
dfx["flip series"] = dfx["genome series"] * (dfx["Root ID"] == 1)
dfx["Competitor"] = (
    dfx
    .groupby(["Stint", "Series", "kind", "Competition Repro"])
    ["flip series"]
    .transform("max")
)
dfx = dfx[
    dfx["Root ID"] == 0
].groupby(["Stint", "Series", "kind", "Competitor"])[
    "Focal Prevalence"
].mean().round().reset_index(drop=False)
dfx


In [ ]:
dfxx["Flagged Advantageous Sites"] = dfxx["Is Less Fit"]
dfxx["Flagged Deleterious Sites"] = dfxx["Is More Fit"]

dfj = dfx.join(
    dfxx[
        [
            "Stint",
            "Series",
            "Flagged Advantageous Sites",
            "Flagged Deleterious Sites",
        ]
    ].set_index(
        ["Stint", "Series"]
    ),
    on=["Stint", "Series"],
    how="inner",
).reset_index(drop=False)
dfj["Favored"] = dfj["Focal Prevalence"] > 0.5
dfj


In [ ]:
light_palette=[
    "#a3cf73",
    "#6dceb1",
]
dark_palette=[
    "#4a772f",
    "#137177",
]


In [ ]:
import itertools as it

for (y, when, kind) in it.product(
    ["Focal Prevalence"],
    ["early", "late", "all"],
    ["eco", "self", "none"],
):
    # 1. Base Aggregation
    dfjx = dfj.dropna().groupby(
        ["Series", "kind", "Stint"],
    ).mean().reset_index(drop=False)

    # 2. Separate Baseline (eco) and Target (self)
    # Note: Using 'eco' as baseline per your snippet
    df_kind = dfjx[dfjx["kind"] == kind].copy()

    df_diff = df_kind

    # 4. Calculate Difference
    df_diff["Genotype Complexity"] = df_diff["Flagged Advantageous Sites"]

    # 5. Prepare Plot Data (Aggregated)
    whenisin = {
        "early": range(0, 51, 10),
        "late": range(60, 101, 10),
        "all": range(0, 101, 10),
    }[when]
    df_plot = df_diff[
        df_diff["Stint"].isin(whenisin)
    ].groupby(["Series"]).mean().reset_index(drop=False)

    # 6. Calculate Regression Stats
    slope, intercept, r_value, p_value, std_err = scipy_stats.linregress(
        df_plot["Genotype Complexity"],
        df_plot[y]
    )

    # Determine significance
    is_significant = p_value < 0.05

    # Format the annotation text with bold if significant
    if is_significant:
        stats_text = (
            f"$\\mathbf{{R^2 = {r_value**2:.2f}}}$\n"
            f"$\\mathbf{{p = {p_value:{'.1e' if p_value < 0.01 else '.3f'}}}}$"
        )
    else:
        stats_text = (
            f"$R^2 = {r_value**2:.2f}$\n"
            f"$p = {p_value:{'.1e' if p_value < 0.01 else '.3f'}}$"
        )

    # 7. Plot
    with tp.teed(
        sns.lmplot,
        data=df_plot,
        x="Genotype Complexity",
        y=y,
        line_kws={"lw": 1.0, "alpha": 0.7, "color": "C0", "linestyle": '-' if is_significant else ':'},
        facet_kws=dict(
            sharex=False,
            sharey=True,
        ),
        scatter_kws={"alpha": 0.3, "clip_on": False, "s": 5, "color": "steelblue"},
        teeplot_outattrs={"when": when},
        teeplot_subdir=teeplot_subdir,
    ) as g:
        g.figure.set_size_inches(2, 1.2)  # Adjusted size for single plot
        g.set_xlabels("Genotype Complexity")
        g.set_ylabels("")

        # Add a horizontal reference line
        ax = g.axes.flat[0]
        ax.axhline(0, color="k", ls="--", lw=0.5, alpha=0.5)
        # ax.set_ylim(-0.18, 0.42)

        # Add the stats annotation to the plot
        # Anchored to upper left or right depending on data layout
        g.figure.subplots_adjust(left=0.2)
        ax.text(
            1, 0,
            stats_text,
            transform=ax.transAxes,
            horizontalalignment='right',
            verticalalignment='bottom',
            fontsize=8,
            color="black" if is_significant else "lightgray",
            weight="bold" if is_significant else "normal",
        )
